# Downstream Impact of OOD Detection

The following demonstrates how the downstream performance impact of OOD detection was evaluated. We assume model training and evaluation has already been run (see notebook 01) and use the sampling rate perturbation at strength 200 Hz as example.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from math import ceil
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

from oddeeg.utils import construct_results_path
from oddeeg.aggregators import KDE, MaxQuantile

In [2]:
perturbation = "perturbation_sfreq"
strength = "200"

dsc_train_cfg = {
    "dataset_name": "TUAB",
    "task": "normality",
    "model_name": "TCN",
    "training_mode": "discriminative"
}

gen_train_cfg = {
    "dataset_name": "TUAB",
    "task": None,  # unconditional generative model
    "model_name": "UNet",
    "training_mode": "flow_matching",
    "max_batches": 1000000,
}

In [3]:
METRICS = {
    "msp": {"label": "MSP", "higher_is_ood": False},
    "energy": {"label": "energy", "higher_is_ood": True},
    "odin": {"label": "ODIN", "higher_is_ood": False},
    "ash": {"label": "ASH", "higher_is_ood": True},
    "log_likelihood": {"label": "Log Likelihood", "higher_is_ood": False},
    "typicality": {"label": "Typicality", "higher_is_ood": True},
    "dose": {"label": "DoSE", "higher_is_ood": False},
    "sitn": {"label": "SITN", "higher_is_ood": True},
}

PERTURBATION_META = {
    "perturbation_sfreq": {"id_sentinel": 100},
    "perturbation_channel_shuffle": {"id_sentinel": 0},
    "perturbation_highpass_hz": {"id_sentinel": 0.0},
    "perturbation_lowpass_hz": {"id_sentinel": float("inf")},
    "perturbation_reref_scheme": {"id_sentinel": "average"},
}

In [ ]:
# Fit OOD detection methods using in-distribution validation data
preds_id_val_gen = pd.read_csv(construct_results_path(config=gen_train_cfg, split_pick="val"))
preds_id_val_dsc = pd.read_csv(construct_results_path(config=dsc_train_cfg, split_pick="val"))
preds_id_val = preds_id_val_gen.merge(preds_id_val_dsc)

if "typicality" in METRICS:
    entropy_estimate = np.mean(preds_id_val["log_likelihood"])
if "dose" in METRICS:
    dose = KDE(features=["log_likelihood", "source_log_likelihood", "log_determinant"])
    dose.fit(preds_id_val, subsample=10000)
if "sitn" in METRICS:
    sitn = MaxQuantile({"anderson_darling_statistic": True, "ps_cv": True})
    sitn.fit(preds_id_val)

In [5]:
id_sentinel = PERTURBATION_META.get(perturbation, {}).get("id_sentinel", 0)

# Load ID test preds
preds_id_test_gen = pd.read_csv(construct_results_path(config=gen_train_cfg, split_pick="test"))
preds_id_test_dsc = pd.read_csv(construct_results_path(config=dsc_train_cfg, split_pick="test"))
preds_id = preds_id_test_gen.merge(preds_id_test_dsc)
preds_id[perturbation] = id_sentinel

# Load OOD test preds
results_path = construct_results_path(config=gen_train_cfg, split_pick="test", **{perturbation: strength})
if not results_path.exists():
    raise FileNotFoundError(f"Results not found: {results_path}")
preds_ood_gen = pd.read_csv(results_path)

results_path = construct_results_path(config=dsc_train_cfg, split_pick="test", **{perturbation: strength})
if not results_path.exists():
    raise FileNotFoundError(f"Results not found: {results_path}")
preds_ood_dsc = pd.read_csv(results_path)

preds_ood = preds_ood_gen.merge(preds_ood_dsc)
preds_ood[perturbation] = strength

# Combine ID and OOD preds
preds = pd.concat([preds_id, preds_ood], ignore_index=True)
preds["correct"] = preds["target"] == preds["pred"]
preds["ood"] = preds[perturbation] != id_sentinel

# Add OOD scores with fitted methods
if "typicality" in METRICS:
    preds["typicality"] = (preds["log_likelihood"] - entropy_estimate).abs()
if "dose" in METRICS:
    preds["dose"] = dose.score(preds)
if "sitn" in METRICS:
    preds["sitn"] = sitn.score(preds)

# Update ID and OOD preds after adding new metrics
preds_id = preds[~preds["ood"]]
preds_ood = preds[preds["ood"]]

### Downstream performance on pooled in- and out-of-distribution data

In [ ]:
performance = "accuracy"  # or "balanced_accuracy"
quantile_edges = [0, 0.001, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.999, 1.0]

results = []
for metric_key, metric_info in METRICS.items():
    if metric_key not in preds.columns:
        continue
        
    metric_label = metric_info.get("label", metric_key)
    
    # Calculate threshold values for the quantiles
    bin_edges = preds[metric_key].quantile(quantile_edges).values

    # Compute performance within bins
    for i in range(len(bin_edges) - 1):
        # Define the subset mask for the current bin
        lower = preds[metric_key] >= bin_edges[i]
        upper = preds[metric_key] <= bin_edges[i + 1] if i == len(bin_edges) - 2 else preds[metric_key] < bin_edges[i + 1]
        subset = preds[lower & upper]
        
        # Compute performance
        if len(subset) > 0:
            if performance == "accuracy":
                metric_val = subset["correct"].mean()
            elif performance == "balanced_accuracy":
                if len(subset["target"].unique()) >= 2:
                    metric_val = balanced_accuracy_score(subset["target"], subset["pred"])
                else:
                    metric_val = np.nan
        else:
            metric_val = np.nan
            
        # Compute counts
        total_count = len(subset)
        id_count = (~subset["ood"]).sum()
        ood_count = subset["ood"].sum()
        
        # Format bin labels
        q_start = quantile_edges[i] * 100
        q_end = quantile_edges[i + 1] * 100
        bin_label = f"[{q_start:.3g}, {q_end:.3g})" if i < len(bin_edges) - 2 else f"[{q_start:.3g}, {q_end:.3g}]"
        
        results.append({
            "Metric": metric_label,
            "Quantile Bin (%)": bin_label,
            f"{performance.replace('_', ' ').title()} (%)": metric_val * 100 if not np.isnan(metric_val) else np.nan,
            "ID Samples": id_count,
            "OOD Samples": ood_count,
            "Total Samples": total_count,
        })
        
df_quantiles = pd.DataFrame(results)
df_quantiles

### Downstream performance in-distribution data only

In [ ]:
results_id = []
for metric_key, metric_info in METRICS.items():
    if metric_key not in preds_id.columns:
        continue
        
    metric_label = metric_info.get("label", metric_key)
    
    # Calculate the actual threshold values for the quantiles
    bin_edges = preds_id[metric_key].quantile(quantile_edges).values

    # Compute performance within bins
    for i in range(len(bin_edges) - 1):
        lower = preds_id[metric_key] >= bin_edges[i]
        upper = preds_id[metric_key] <= bin_edges[i + 1] if i == len(bin_edges) - 2 else preds_id[metric_key] < bin_edges[i + 1]
        subset = preds_id[lower & upper]
        
        # Compute performance
        if len(subset) > 0:
            if performance == "accuracy":
                metric_val = subset["correct"].mean()
            elif performance == "balanced_accuracy":
                if len(subset["target"].unique()) >= 2:
                    metric_val = balanced_accuracy_score(subset["target"], subset["pred"])
                else:
                    metric_val = np.nan
        else:
            metric_val = np.nan
            
        # Format bin labels nicely
        q_start = quantile_edges[i] * 100
        q_end = quantile_edges[i + 1] * 100
        bin_label = f"[{q_start:.3g}, {q_end:.3g})" if i < len(bin_edges) - 2 else f"[{q_start:.3g}, {q_end:.3g}]"
        
        results_id.append({
            "Metric": metric_label,
            "Quantile Bin (%)": bin_label,
            f"{performance.replace('_', ' ').title()} (%)": metric_val * 100 if not np.isnan(metric_val) else np.nan,
            "Total Samples": len(subset),
        })
        
df_quantiles_id = pd.DataFrame(results_id)
df_quantiles_id